## **Scratchpad**


- Survey is Mostly Qualitative
    - Either categorical / rank scale (1-5)
- So many questions that can be asked / answered
- First thought was correlation between a variety of responses and year / program  / demography
    - Correlation within responses
- Histograms will be your friend

---

## Codebook read-in


- There are probably a few ways to hit this, there may be encoding in an Excel tab
- Could Parse out the text from the PDF
   - Regex but will be messy
    - Gemini / Claude API for one-shot NLP?

After spending some time fiddling with a Regex search,  I think 

### **Scope is Graduate School Leadership**

- I think start by scoping the potential types of questions:

  - Financial
  - Academic by:
    - Program
    - Year
  - Demographic?  Doesn't seem there are strong demographic indicators in this set
    - Response bias / survey bias



## **Primary Analysis**

> This analysis focuses on the use of AI by graduate program. The goal is to understand how AI impacts perception of learning outcomes.

- The primary dimension is Program, with splits by Social Sciencees / 

In [35]:
#  --- Using env_finance see environment.yml for quick pacakge installation

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from IPython.display import display, Markdown


In [36]:
# Data read-ins and directory setup

directory = os.getcwd()

raw_survey = pd.read_excel("/mnt/hard_storage/GithubRepos/project-dev/Ad_Hoc/GradSurvey/CU Boulder SERU dataset_sample.xlsx")

## **Data Dictionary**

> See the conversation at https://claude.ai/share/1f6c5ed3-f888-4b40-8ab4-3780ffc70163 for full prompting

The actual codebook is not cleanly formatted or tabular. Rather than relying on manual updating of data names and memorizing data while working, using Claude to build Regex matching to extract the survey questions and compile them into a dictionary corresponding to the variable names.

---

####  **Naming Patterns**

- Can use `()`, `CaSe`, and `?` as general delimiters for the search

- Consisten Inconsistencies:
    - Standalone questions put the code before the stem: `GS0401_GSADPRIMRYAD` Do you currently have a primary advisor?
    - Matrix items put it after the item text, often on a wrapped line: Quality of instruction \n (`GS1101_GSOSQLTYINST`)
    - Matrix stems carry no code eg,
        - Question ("To what extent are you satisfied...") is only recoverable by looking upward past the response-scale header row


#### Quirks _(Caught by Claude)_

- Duplicate codes: `GS0301_GSFSCURRCHAS_R22` appears on both "Research assistantship" and "Other assistantship." `GS1302_GSGGAI_9` appears twice. Your parser should surface collisions, not silently dedupe.
- Inconsistent suffix casing: `_R22` vs `_r20` vs `_n20` vs `_r19`. Case-insensitive on the suffix, case-sensitive on the stem.
- Codes without the GS prefix: `CPCURINDT_r20`, `INT0004`, `GSADHLPDSDEF`, `GSTETRNMLTCL`.
- Prefix-only codes attached to matrix stems: `GS1303_GSGAIPDC` sits inline after the item text with no parens

## **EDA**
#### The following blocks:

 - Conduct basic EDA _(exploratory data analysis)_ using standard methods:
    - `.head()`  - First 10 entries of the dataset
    - `.describe()` - Summary statistics for each variable (column)
    - `.info()` - Data types / metadata  / general information
    - `.shape()`  - The number of rows and columns as `(Rows x Columns)`

 - Look for common data hiccups like:
    - Missing values
    - Incorrect / broken variable types
    - 

---

In [37]:
# --- Tabular description of the survey dataset
 
display(Markdown("## First 5 rows of the survey dataset"))
display(raw_survey.head())
print("\n")
display(Markdown("---"))
print("\n")
display(Markdown("## Survey Summary Statistics"))
display(raw_survey.describe())
display(Markdown("---"))


## First 5 rows of the survey dataset

,PROGRESS,DURATION,FINISHED,CONSENT,GS0101_GSYPPROGNAME,GS0102_GSYPSPECPROG,GS0103B_GSYPSPECSTRT,GS0104_GSYPTAKECRSE_r23,GS0104_GSYPWORKDISS_r23,GS0104_GSYPDEFDDISS_r23,...,CIP_CODE1,COLLEGE_CODE1,COLLEGE_NAME1,LEVEL_GRAD,LOCATION,PROGRAM_CODE1,PROGRAM_TEXT1,YEAR,TERM1,CIP_CODE2010
0,100,1668,True,Agree,Program name (Seed file),"Educ Foundations, Pol & Prac",Fourth year,Checked,Not Checked,Not Checked,...,130901,EDUC,School of Education,1,Boulder,EFPP,"Educ Foundations, Pol & Prac",2021.0,Summer,130901.0
1,100,1677,True,Agree,Program name (Seed file),Physics,Sixth year or above,Not Checked,Checked,Not Checked,...,400801,ARSC,College of Arts & Sciences,3,Boulder,PHYS,Physics,2019.0,Fall,400801.0
2,100,94906,True,Agree,Program name (Seed file),Electrical Engineering,First year,Checked,Not Checked,Not Checked,...,141001,ENGR,College of Engr & Applied Sci,3,Boulder,EEEN,Electrical Engineering,2024.0,Fall,141001.0
3,100,1334,True,Agree,Program name (Seed file),Astrophysical & Planetary Sci,Third year,Checked,Checked,Not Checked,...,400202,ARSC,College of Arts & Sciences,3,Boulder,ASPS,Astrophysical & Planetary Sci,2022.0,Fall,400202.0
4,22,541,False,Agree,Program name (Seed file),Electrical Engineering,Fourth year,Not Checked,Checked,Not Checked,...,141001,ENGR,College of Engr & Applied Sci,3,Boulder,EEEN,Electrical Engineering,2021.0,Fall,141001.0


---

## Survey Summary Statistics

,PROGRESS,DURATION,GS1208_GSDMCHLD0TO1_2,GS1208_GSDMCHLD0TO1_3,CIP_CODE1,LEVEL_GRAD,YEAR,CIP_CODE2010
count,1387.000000,1.387000e+03,101.000000,101.000000,1387.000000,1387.000000,1206.000000,1207.000000
mean,87.731795,2.337430e+05,0.108911,0.594059,258498.767123,2.532805,2022.137645,257793.391881
std,28.091330,7.185685e+05,1.066780,1.209779,142780.814463,0.854180,2.108105,143460.777401
min,11.000000,3.900000e+01,-1.000000,-1.000000,30103.000000,1.000000,2005.000000,30103.000000
25%,100.000000,1.119000e+03,-1.000000,-1.000000,140401.000000,3.000000,2021.000000,140401.000000
50%,100.000000,1.680000e+03,0.000000,1.000000,230101.000000,3.000000,2023.000000,230101.000000
75%,100.000000,5.640000e+03,1.000000,2.000000,400601.000000,3.000000,2024.000000,400601.000000
max,100.000000,4.496931e+06,3.000000,3.000000,540101.000000,4.000000,2025.000000,540101.000000


---

In [38]:
# --- General Info about the survey dataset

display(Markdown("## Survey Info"))
raw_survey.info() 
display(Markdown("---"))
display(Markdown(f"## Survey Shape: `{raw_survey.shape}`"))
display(Markdown("---"))
#display(Markdown(f"## Survey Variables:"))
#display(Markdown("\n".join(f"- `{c}`" for c in raw_survey.columns)))


## Survey Info

<class 'pandas.DataFrame'>
RangeIndex: 1387 entries, 0 to 1386
Columns: 289 entries, PROGRESS to CIP_CODE2010
dtypes: bool(1), float64(4), int64(4), object(252), str(28)
memory usage: 3.4+ MB


---

## Survey Shape: `(1387, 289)`

---

In [39]:
# --- Looking for potential missing values in the survey dataset

missing_data = {
    "COLNAME": [],
    "DTYPE": [],
    "NMISSING": []
}

# Using Pandas Indexing to Identify Missing Values in the Survey Dataset

missing_data = pd.DataFrame({

    "Data Type": raw_survey.dtypes,
    "Quantity Missing": raw_survey.isnull().sum(),
})

missing_data = missing_data.rename_axis("Variable").reset_index()
missing_data = missing_data[missing_data["Quantity Missing"] > 0]

display(Markdown("## Missing Values in the Survey Dataset"))
display(missing_data)



display(Markdown("---"))

## Missing Values in the Survey Dataset

,Variable,Data Type,Quantity Missing
16,GS1101_GSOSQLTYINST,object,41
17,GS1101_GSOSCRSEAVLB,object,41
18,GS1101_GSOSQLTYADVS,object,41
19,GS1101_GSOSAVLBADVS,object,41
20,GS1101_GSOSKNWLGAIN,object,41
...,...,...,...
277,INT0007_ISHAROT_n21,object,1328
278,INT0007_ISHARNONE_n21,object,1328
286,YEAR,float64,181
287,TERM1,str,181


---

In [40]:
# === Basic Cleaning 

# Drop where all values are missing
raw_survey = raw_survey.dropna(how='all')

### Patterns In the Missing Data

1. There is data systematically missing
     - Counts reveal clustering about matrix questions

## **Data Dictionary**

This dictionary allows quick mapping of survey columns to the content of the question 

---

The data dictionary provided (_gradSERU 2025 Survey Instrument_sample.pdf_) is a backend survey export. Based on reading it, it seems that extracting a tabular form of question codes, and their corresponding information isn't straightforward. Because of that, I used Claude 3.5 Opus to create a comprehensive regular expression to extract question text, sub content, ranges of Likert scales, and other information.

The reason for doing this, rather than manually selecting a sub-section of chosen information is primarily because of the nature of the analysis: the outcome dimensions are not exclusive to one question or section or the survey. The full regular expression can be found in the `/GradSurvey` directory

In [41]:
# === Data Dictionary

data_dictionary = pd.read_csv(directory +"/seru_data_dictionary.csv")

data_dictionary[["prefix","sub_question", "suffix"]] = data_dictionary['survey_column'].str.split("_", n=2, expand=True)

data_dictionary['section_code']  = data_dictionary['sub_question'].str[:4]
data_dictionary['sub_question']  = data_dictionary['sub_question'].str[4:]



In [42]:
#  ---  Semi manual generation of the Variable Description Table

# Splitting varnames for searching questions and matrix sub-questions

question_data = raw_survey.columns.str.split("_")
varnames = pd.DataFrame(question_data.tolist(), columns=["number", "sub_question", "suffix"])

varnames['section']  = varnames['sub_question'].str[:4]
varnames['sub_question']  = varnames['sub_question'].str[4:]
varnames = varnames.rename(columns={"number": "question_number"})  

In [43]:
# === AI Use Subquestions

ai_use_codes = ["GS1301_GSGAIDAY",
                "GS1303_GSGAIPDC",
                "GS1303_GSGAIAUC",
                "GS1303_GSGAIGPT",
                "GS1303_GSGAIGURW",
                "GS1303_GSGAIGR",
                "GS1303_GSGAIEPDR"]

gen_ai_use = raw_survey[ai_use_codes].copy()

ai_use_mapping = (
    data_dictionary[data_dictionary["survey_column"].isin(ai_use_codes)]
    .set_index("survey_column")[["question_stem", "item_text"]]
    .to_dict(orient="index")
)

print("AI Use Mapping:")
print(ai_use_mapping)

AI Use Mapping:
{'GS1301_GSGAIDAY': {'question_stem': 'Generative artificial intelligence (AI) tools like ChatGPT are becoming more common in academic settings During this academic year, how often have you used such tools?', 'item_text': 'Generative AI hover over definition: Generative artificial intelligence (AI) refers to computer programs that can generate text, images, or other content based on provided prompts (e.g., ChatGPT, Google Bard, or Microsoft Bing Chat)'}, 'GS1303_GSGAIPDC': {'question_stem': 'To what extent do you agree or disagree with the following statements about the use of generative AI tools like ChatGPT', 'item_text': 'My professors have discussed when it is appropriate to use AI to complete my coursework'}, 'GS1303_GSGAIAUC': {'question_stem': 'To what extent do you agree or disagree with the following statements about the use of generative AI tools like ChatGPT', 'item_text': 'I understand when it is appropriate to use AI to complete my coursework'}, 'GS1303_GSG